# Tools

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

Use a decorator to transfer a python function to LLM tool

In [ ]:
from langchain_core.tools import tool
 
@tool   # This funciton will be used as a tool
def test(query: str) -> str:
    """Test function"""
    return "42f"
 
print(f"test.name:{test.name}")
print(f"test.description:{test.description}")
print(f"test.args:{test.args}")

test.name:test
test.description:Test function
test.args:{'query': {'title': 'Query', 'type': 'string'}}


In [3]:
test.run("a")

'42f'

In [5]:
from pydantic import BaseModel, Field
class SearchInput(BaseModel):
    query: str = Field(description="Thing to search for")

@tool(args_schema=SearchInput)  
def test(query: str) -> str:
    """Test function"""
    return "42f"

print(f"test.args:{test.args}")

test.args:{'query': {'description': 'Thing to search for', 'title': 'Query', 'type': 'string'}}


In [6]:
import requests

@tool
def getweather(location: str) -> str:
    '''
    Fetch current weather for given location
    Args: location(str)
    Returns: str: Text about weather condition
    '''
    url="https://uapis.cn/api/v1/misc/weather"
    try:
        response=requests.get(url, params={"city":location}, timeout=5)
        response.raise_for_status()
        data=response.json()
        city_name=data.get("city", location)
        weather=data.get("weather", "")
        temperature=data.get("temperature", "")
        wind_direction=data.get("wind_direction", "")
        wind_power=data.get("wind_power", "")
        humidity=data.get("humidity", "")
        report_time=data.get("report_time", "")
        
        result = (f"{city_name} current weather: {weather}, {temperature}C\n"
          f"{wind_direction}{wind_power}, humidity {humidity}% ({report_time})")
        return result
    except requests.exceptions.Timeout:
        return "TimeoutError"
    except Exception as e:
        return f"error {e}"

In [7]:
getweather.run("Shanghai")

'上海 current weather: 晴, 30C\n东北风2级, humidity 71% (9 分钟前发布)'

## Langgraph

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from langchain.agents import create_agent

agent=create_agent(model="deepseek-chat", tools=[getweather])

#   Streaming Message Chunks
inputs = {"messages": [HumanMessage(content="What's the weather like in Shanghai")]}

for token, output in agent.stream(inputs, stream_mode="messages"):  # Only message
    print(token.content, end="", flush=True)    #   Output token by token
   

Let me check the current weather in Shanghai for you.上海 current weather: 晴, 30C
西北风2级, humidity 73% (9 分钟前发布)The current weather in Shanghai is:

🌤 **Sunny (晴)**
- **Temperature**: 30°C
- **Wind**: Northwest wind, light breeze (level 2)
- **Humidity**: 73%

It's a warm and sunny day in Shanghai! ☀️

The graph structure is a ring. After the data enters the tool node (getweather) and finishes execution, it will loop back to the large language model node. The model receives the weather string and proceeds to the second round of reasoning.

## Langchain

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="deepseek-chat", temperature=0,
                  api_key=os.getenv("DEEPSEEK_API_KEY"),
                  base_url=os.getenv("DEEPSEEK_BASE_URL"))

model_with_tool = model.bind_tools([getweather], tool_choice="getweather")
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful weather assistant."),
    ("human", "{input}")
])


In [ ]:
weather_chain=prompt|model_with_tool
result = weather_chain.invoke({"input": "What's the weather like in Shanghai?"})
print(result)   # Return AImessage with tool calls, but did not run the tool

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 308, 'total_tokens': 345, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 308}, 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'a6587563-9e8b-42f7-b9fc-d1bb060ed33b', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f4cd9-d767-72c2-a46f-c1e32fd892ed-0' tool_calls=[{'name': 'getweather', 'args': {'location': 'Shanghai'}, 'id': 'call_00_z2kNAkgZRDEtHbnqHg1M7390', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 308, 'output_tokens': 37, 'total_tokens': 345, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [12]:
def call_weather_tool(ai_message):
    tool_call = ai_message.tool_calls[0] 
    args = tool_call["args"] 
    return getweather.invoke(args)

weather_chain=prompt|model_with_tool|RunnableLambda(call_weather_tool)  # Run getweather function
result = weather_chain.invoke({"input": "What's the weather like in Shanghai?"})
print(result) 

上海 current weather: 多云, 28C
东南风2级, humidity 83% (9 分钟前发布)
